# Pandas에서 데이터 작업

NumPy의 장점 중 하나는 기본 산술(덧셈, 뺄셈, 곱셈 등)과 더 복잡한 연산(삼각 함수, 지수 및 로그 함수 등)을 사용하여 요소별 연산을 빠르게 수행할 수 있다는 것입니다.
Pandas는 NumPy에서 이 기능의 대부분을 상속하며 [NumPy 배열에 대한 계산: 범용 함수](02.03-Computation-on-arrays-ufuncs.ipynb)에 소개된 ufuncs가 이에 대한 핵심입니다.

그러나 Pandas에는 몇 가지 유용한 변형이 포함되어 있습니다. 부정 및 삼각 함수와 같은 단항 연산의 경우 이러한 ufunc는 출력에서 ​​*인덱스 및 열 레이블*을 유지하며, 더하기 및 곱셈과 같은 이진 연산의 경우 Pandas는 객체를 ufunc에 전달할 때 자동으로 *인덱스를 정렬*합니다.
이는 데이터의 컨텍스트를 유지하고 다양한 소스의 데이터를 결합하는 것(원시 NumPy 배열을 사용하여 잠재적으로 오류가 발생하기 쉬운 작업)이 Pandas를 사용하면 본질적으로 완벽하다는 것을 의미합니다.
추가적으로 1차원 `Series` 구조와 2차원 `DataFrame` 구조 사이에 잘 ​​정의된 작업이 있다는 것도 살펴보겠습니다.

## Ufuncs: 인덱스 보존

Pandas는 NumPy와 함께 작동하도록 설계되었으므로 모든 NumPy ufunc는 Pandas 'Series' 및 'DataFrame' 개체에서 작동합니다.
이를 보여주기 위해 간단한 `Series`와 `DataFrame`을 정의하는 것부터 시작해 보겠습니다.

In [1]:
import pandas as pd
import numpy as np

In [2]:
rng = np.random.default_rng(42)
ser = pd.Series(rng.integers(0, 10, 4))
ser

0    0
1    7
2    6
3    4
dtype: int64

In [3]:
df = pd.DataFrame(rng.integers(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
0,4,8,0,6
1,2,0,5,9
2,7,7,7,7


이 개체 중 하나에 NumPy ufunc를 적용하면 결과는 *인덱스가 보존된 다른 Pandas 개체가 됩니다.*

In [4]:
np.exp(ser)

0       1.000000
1    1096.633158
2     403.428793
3      54.598150
dtype: float64

이는 더 복잡한 작업 시퀀스에도 적용됩니다.

In [5]:
np.sin(df * np.pi / 4)

,A,B,C,D
0,1.224647e-16,-2.449294e-16,0.000000,-1.000000
1,1.000000e+00,0.000000e+00,-0.707107,0.707107
2,-7.071068e-01,-7.071068e-01,-0.707107,-0.707107


[NumPy 배열 계산: 범용 함수](02.03-Computation-on-arrays-ufuncs.ipynb)에서 논의된 모든 ufunc는 유사한 방식으로 사용될 수 있습니다.

## Ufuncs: 인덱스 정렬

두 개의 'Series' 또는 'DataFrame' 개체에 대한 이진 작업의 경우 Pandas는 작업을 수행하는 과정에서 인덱스를 정렬합니다.
이는 다음 몇 가지 예에서 볼 수 있듯이 불완전한 데이터로 작업할 때 매우 편리합니다.

### 시리즈 인덱스 정렬

예를 들어 두 개의 서로 다른 데이터 소스를 결합하고 *지역* 기준으로 미국 상위 3개 주만 찾고 *인구* 기준으로 미국 상위 3개 주만 찾으려고 한다고 가정해 보겠습니다.

In [6]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 39538223, 'Texas': 29145505,
                        'Florida': 21538187}, name='population')

인구 밀도를 계산하기 위해 이들을 나누면 어떤 일이 발생하는지 봅시다.

In [7]:
population / area

Alaska              NaN
California    93.257784
Florida             NaN
Texas         41.896072
dtype: float64

결과 배열에는 두 입력 배열의 인덱스 *합집합*이 포함되며, 이는 다음 인덱스에서 직접 확인할 수 있습니다.

In [8]:
area.index.union(population.index)

Index(['Alaska', 'California', 'Florida', 'Texas'], dtype='object')

항목이 없는 항목은 `NaN` 또는 "숫자가 아님"으로 표시됩니다. 이는 Pandas가 누락된 데이터를 표시하는 방법입니다([누락된 데이터 처리](03.04-Missing-Values.ipynb)에서 누락된 데이터에 대한 자세한 설명 참조).
이 인덱스 일치는 파이썬(Python)의 내장 산술 표현식에 대해 이러한 방식으로 구현됩니다. 누락된 값은 `NaN`으로 표시됩니다.

In [9]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
A + B

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

NaN 값을 사용하는 것이 원하는 동작이 아닌 경우 연산자 대신 적절한 개체 메서드를 사용하여 채우기 값을 수정할 수 있습니다.
예를 들어 ``A.add(B)`` 호출은 ``A + B`` 호출과 동일하지만 ``A`` 또는 ``B``에서 누락될 수 있는 요소에 대한 채우기 값을 선택적으로 명시적으로 지정할 수 있습니다.

In [10]:
A.add(B, fill_value=0)

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

### DataFrames의 인덱스 정렬

`DataFrame` 객체에 대한 작업을 수행할 때 열과 인덱스 *모두*에 대해 유사한 유형의 정렬이 발생합니다.

In [11]:
A = pd.DataFrame(rng.integers(0, 20, (2, 2)),
                 columns=['a', 'b'])
A

,a,b
0,10,2
1,16,9


In [12]:
B = pd.DataFrame(rng.integers(0, 10, (3, 3)),
                 columns=['b', 'a', 'c'])
B

,b,a,c
0,5,3,1
1,9,7,6
2,4,8,5


In [13]:
A + B

,a,b,c
0,13.0,7.0,NaN
1,23.0,18.0,NaN
2,NaN,NaN,NaN


두 개체의 순서에 관계없이 인덱스가 올바르게 정렬되고 결과의 인덱스가 정렬됩니다.
'Series'의 경우와 마찬가지로 관련 개체의 산술 메서드를 사용하고 누락된 항목 대신 사용할 원하는 'fill_value'를 전달할 수 있습니다.
여기에서는 `A`에 있는 모든 값의 평균을 채울 것입니다.

In [14]:
A.add(B, fill_value=A.values.mean())

,a,b,c
0,13.00,7.00,10.25
1,23.00,18.00,15.25
2,17.25,13.25,14.25


다음 표에는 파이썬(Python) 연산자와 이에 상응하는 Pandas 개체 메서드가 나열되어 있습니다.

| 파이썬 연산자 | Pandas 메서드 |
|-----------------|---------------------------------|
| `+` | `추가` |
| `-` | `sub`, `subtract` |
| `*` | `mul`, `곱하기` |
| `/` | `truediv`, `div`, `divide` |
| `//` | `floordiv` |
| `%` | 모드' |
| `**` | '펑' |


## Ufuncs: DataFrame과 시리즈 간의 작업

'DataFrame'과 'Series' 간의 작업을 수행할 때 인덱스와 열 정렬이 유사하게 유지되며 결과는 2차원 및 1차원 NumPy 배열 간의 작업과 유사합니다.
2차원 배열과 해당 행 중 하나의 차이점을 찾는 일반적인 작업을 생각해 보세요.

In [15]:
A = rng.integers(10, size=(3, 4))
A

array([[4, 4, 2, 0],
       [5, 8, 0, 8],
       [8, 2, 6, 1]])

In [16]:
A - A[0]

array([[ 0,  0,  0,  0],
       [ 1,  4, -2,  8],
       [ 4, -2,  4,  1]])

NumPy의 브로드캐스팅 규칙([배열에 대한 계산: 브로드캐스트](02.05-Computation-on-arrays-broadcasting.ipynb) 참조)에 따라 2차원 배열과 해당 행 중 하나 사이의 빼기가 행 단위로 적용됩니다.

Pandas에서는 기본적으로 규칙이 행 단위로 유사하게 작동합니다.

In [17]:
df = pd.DataFrame(A, columns=['Q', 'R', 'S', 'T'])
df - df.iloc[0]

,Q,R,S,T
0,0,0,0,0
1,1,4,-2,8
2,4,-2,4,1


대신 열 단위로 작업하려면 `axis` 키워드를 지정하면서 앞서 언급한 객체 메서드를 사용할 수 있습니다.

In [18]:
df.subtract(df['R'], axis=0)

,Q,R,S,T
0,0,0,-2,-4
1,-3,0,-8,0
2,6,0,4,-1


이전에 설명한 작업과 마찬가지로 이러한 `DataFrame`/`Series` 작업은 두 요소 사이의 인덱스를 자동으로 정렬합니다.

In [19]:
halfrow = df.iloc[0, ::2]
halfrow

Q    4
S    2
Name: 0, dtype: int64

In [20]:
df - halfrow

,Q,R,S,T
0,0.0,NaN,0.0,NaN
1,1.0,NaN,-2.0,NaN
2,4.0,NaN,4.0,NaN


이러한 인덱스와 열의 보존 및 정렬은 Pandas의 데이터 작업이 항상 데이터 컨텍스트를 유지하여 원시 NumPy 배열에서 이종 및/또는 잘못 정렬된 데이터로 작업할 때 발생할 수 있는 일반적인 오류를 방지한다는 것을 의미합니다.